# Multi-Game Stint Construction

This notebook generalizes the original one-game stint reconstruction prototype into a multi-game pipeline.

The goal is to process multiple regular-season games, infer starting lineups automatically when possible, track substitution-based lineup changes, validate lineup integrity, and save a larger stint dataset for modeling.

In [3]:
import sqlite3
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 200)

##  Load multi-game play-by-play sample

We load a regular-season sample of 50 games. Preseason games are excluded because they may contain unusual rotations, international matchups, and noisier roster information.

In [5]:
conn = sqlite3.connect("../data/raw/nba.sqlite")

sample_games = pd.read_sql(
    """
    SELECT DISTINCT game_id
    FROM play_by_play
    WHERE game_id LIKE '002%'
    ORDER BY game_id
    LIMIT 200;
    """,
    conn
)

sample_game_ids = sample_games["game_id"].tolist()

placeholders = ",".join(["?"] * len(sample_game_ids))

pbp_sample = pd.read_sql(
    f"""
    SELECT *
    FROM play_by_play
    WHERE game_id IN ({placeholders})
    ORDER BY game_id, period, eventnum;
    """,
    conn,
    params=sample_game_ids
)

print("Games loaded:", pbp_sample["game_id"].nunique())
print("Dataset shape:", pbp_sample.shape)

pbp_sample.head(10)

Games loaded: 200
Dataset shape: (91628, 34)


,game_id,eventnum,eventmsgtype,eventmsgactiontype,period,wctimestring,pctimestring,homedescription,neutraldescription,visitordescription,score,scoremargin,person1type,player1_id,player1_name,player1_team_id,player1_team_city,player1_team_nickname,player1_team_abbreviation,person2type,player2_id,player2_name,player2_team_id,player2_team_city,player2_team_nickname,player2_team_abbreviation,person3type,player3_id,player3_name,player3_team_id,player3_team_city,player3_team_nickname,player3_team_abbreviation,video_available_flag
0,0020000001,0,12,0,1,12:13 PM,12:00,None,Start of 1st Period (12:13 PM EST),None,None,None,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
1,0020000001,1,10,0,1,12:13 PM,12:00,Jump Ball Camby vs. Ratliff: Tip to Houston,None,None,None,None,4.0,948,Marcus Camby,1610612752.0,New York,Knicks,NYK,5.0,689,Theo Ratliff,1610612755.0,Philadelphia,76ers,PHI,4.0,275,Allan Houston,1610612752.0,New York,Knicks,NYK,0
2,0020000001,2,2,1,1,12:14 PM,11:41,MISS Sprewell 6' Jump Shot,None,Ratliff BLOCK (1 BLK),None,None,4.0,84,Latrell Sprewell,1610612752.0,New York,Knicks,NYK,0.0,0,None,None,None,None,None,5.0,689,Theo Ratliff,1610612755.0,Philadelphia,76ers,PHI,0
3,0020000001,3,4,0,1,12:14 PM,11:40,None,None,76ers Rebound,None,None,3.0,1610612755,None,None,None,None,None,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
4,0020000001,4,6,2,1,12:14 PM,11:29,Camby S.FOUL (P1.T1),None,None,None,None,4.0,948,Marcus Camby,1610612752.0,New York,Knicks,NYK,0.0,0,None,None,None,None,None,1.0,0,None,None,None,None,None,0
5,0020000001,5,3,11,1,12:14 PM,11:29,None,None,Ratliff Free Throw 1 of 2 (1 PTS),1 - 0,-1,5.0,689,Theo Ratliff,1610612755.0,Philadelphia,76ers,PHI,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
6,0020000001,6,3,12,1,12:15 PM,11:29,None,None,MISS Ratliff Free Throw 2 of 2,None,None,5.0,689,Theo Ratliff,1610612755.0,Philadelphia,76ers,PHI,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
7,0020000001,7,4,0,1,12:15 PM,11:28,Ward REBOUND (Off:0 Def:1),None,None,None,None,4.0,369,Charlie Ward,1610612752.0,New York,Knicks,NYK,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
8,0020000001,8,6,2,1,12:15 PM,11:18,None,None,Ratliff S.FOUL (P1.T1),None,None,5.0,689,Theo Ratliff,1610612755.0,Philadelphia,76ers,PHI,0.0,0,None,None,None,None,None,1.0,0,None,None,None,None,None,0
9,0020000001,9,3,11,1,12:16 PM,11:18,Camby Free Throw 1 of 2 (1 PTS),None,None,1 - 1,TIE,4.0,948,Marcus Camby,1610612752.0,New York,Knicks,NYK,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0


## Keep columns required for stint reconstruction

We create a focused play-by-play dataset containing only the fields required to infer teams, parse scores, identify substitutions, and track lineups.

In [6]:
cols_needed = [
    "game_id",
    "eventnum",
    "eventmsgtype",
    "eventmsgactiontype",
    "period",
    "pctimestring",
    "homedescription",
    "visitordescription",
    "neutraldescription",
    "score",
    "scoremargin",
    "player1_id",
    "player1_name",
    "player1_team_id",
    "player2_id",
    "player2_name",
    "player2_team_id",
    "player3_id",
    "player3_name",
    "player3_team_id"
]

pbp = pbp_sample[cols_needed].copy()
pbp.shape

(91628, 20)

##  Standardize identifiers and parse scores

Player and team identifiers are standardized as strings to avoid mismatches caused by numeric formatting.

The score field is also parsed into explicit home and away score columns for stint-level scoring.

In [7]:
def clean_id(x):
    if pd.isna(x):
        return np.nan
    return str(x).replace(".0", "")

id_cols = [
    "game_id",
    "player1_id", "player2_id", "player3_id",
    "player1_team_id", "player2_team_id", "player3_team_id"
]

for col in id_cols:
    pbp[col] = pbp[col].apply(clean_id)


def parse_score(score_str):
    if pd.isna(score_str):
        return (np.nan, np.nan)

    parts = str(score_str).split(" - ")

    if len(parts) != 2:
        return (np.nan, np.nan)

    away_score = int(parts[0])
    home_score = int(parts[1])

    return away_score, home_score


pbp[["away_score", "home_score"]] = pbp["score"].apply(
    lambda x: pd.Series(parse_score(x))
)

pbp[["away_score", "home_score"]] = (
    pbp[["away_score", "home_score"]]
    .ffill()
    .fillna(0)
    .astype(int)
)

pbp.head()

,game_id,eventnum,eventmsgtype,eventmsgactiontype,period,pctimestring,homedescription,visitordescription,neutraldescription,score,scoremargin,player1_id,player1_name,player1_team_id,player2_id,player2_name,player2_team_id,player3_id,player3_name,player3_team_id,away_score,home_score
0,0020000001,0,12,0,1,12:00,None,None,Start of 1st Period (12:13 PM EST),None,None,0,None,NaN,0,None,NaN,0,None,NaN,0,0
1,0020000001,1,10,0,1,12:00,Jump Ball Camby vs. Ratliff: Tip to Houston,None,None,None,None,948,Marcus Camby,1610612752,689,Theo Ratliff,1610612755,275,Allan Houston,1610612752,0,0
2,0020000001,2,2,1,1,11:41,MISS Sprewell 6' Jump Shot,Ratliff BLOCK (1 BLK),None,None,None,84,Latrell Sprewell,1610612752,0,None,NaN,689,Theo Ratliff,1610612755,0,0
3,0020000001,3,4,0,1,11:40,None,76ers Rebound,None,None,None,1610612755,None,NaN,0,None,NaN,0,None,NaN,0,0
4,0020000001,4,6,2,1,11:29,Camby S.FOUL (P1.T1),None,None,None,None,948,Marcus Camby,1610612752,0,None,NaN,0,None,NaN,0,0


##  Build player-name lookup

A player-name lookup is built directly from the play-by-play event fields. This avoids relying only on reference tables and improves coverage for players appearing in the selected sample.

In [8]:
def build_player_name_map(df):
    lookup_1 = df[["player1_id", "player1_name"]].dropna().rename(
        columns={"player1_id": "id", "player1_name": "full_name"}
    )

    lookup_2 = df[["player2_id", "player2_name"]].dropna().rename(
        columns={"player2_id": "id", "player2_name": "full_name"}
    )

    lookup_3 = df[["player3_id", "player3_name"]].dropna().rename(
        columns={"player3_id": "id", "player3_name": "full_name"}
    )

    lookup = pd.concat([lookup_1, lookup_2, lookup_3], axis=0).drop_duplicates()
    lookup["id"] = lookup["id"].astype(str)

    return dict(zip(lookup["id"], lookup["full_name"]))


player_name_map = build_player_name_map(pbp)


def lineup_ids_to_names(lineup_ids):
    return [player_name_map.get(str(pid), str(pid)) for pid in lineup_ids]

##  Define reusable helper functions

To scale the stint engine across multiple games, the one-game logic is converted into reusable functions.

Each game is processed independently so that team inference, starter inference, substitutions, and validation are isolated at the game level.

In [9]:
def infer_home_away_team_ids(game_df):
    home_candidates = game_df.loc[
        game_df["homedescription"].notna() & game_df["player1_team_id"].notna(),
        "player1_team_id"
    ]

    away_candidates = game_df.loc[
        game_df["visitordescription"].notna() & game_df["player1_team_id"].notna(),
        "player1_team_id"
    ]

    if home_candidates.empty or away_candidates.empty:
        return None, None

    home_team_id = home_candidates.mode().iloc[0]
    away_team_id = away_candidates.mode().iloc[0]

    if home_team_id == away_team_id:
        return None, None

    return str(home_team_id), str(away_team_id)


def unique_players_from_events(df, team_id):
    players = set()

    for player_col, team_col in [
        ("player1_id", "player1_team_id"),
        ("player2_id", "player2_team_id"),
        ("player3_id", "player3_team_id")
    ]:
        temp = (
            df.loc[df[team_col].astype(str) == str(team_id), player_col]
            .dropna()
            .astype(str)
            .tolist()
        )
        players.update(temp)

    return players


def extract_substitutions(game_df):
    subs = game_df[game_df["eventmsgtype"] == 8].copy()

    subs["sub_team_id"] = subs["player1_team_id"].astype(str)
    subs["player_out_id"] = subs["player1_id"].astype(str)
    subs["player_in_id"] = subs["player2_id"].astype(str)

    return subs


def infer_starters(game_df, home_team_id, away_team_id):
    period_1 = game_df[game_df["period"] == 1].copy()
    early_events = period_1.head(80)

    subs = extract_substitutions(game_df)
    subs_p1 = subs[subs["period"] == 1].copy()

    players_subbed_out_p1 = (
        subs_p1.groupby("sub_team_id")["player_out_id"]
        .apply(lambda x: set(x.dropna().astype(str)))
        .to_dict()
    )

    home_early_players = unique_players_from_events(early_events, home_team_id)
    away_early_players = unique_players_from_events(early_events, away_team_id)

    home_subbed_out = set(players_subbed_out_p1.get(str(home_team_id), []))
    away_subbed_out = set(players_subbed_out_p1.get(str(away_team_id), []))

    home_candidates = home_early_players.union(home_subbed_out)
    away_candidates = away_early_players.union(away_subbed_out)

    if len(home_candidates) < 5 or len(away_candidates) < 5:
        return None, None, "insufficient_candidates"

    home_starters = sorted(list(home_candidates))[:5]
    away_starters = sorted(list(away_candidates))[:5]

    return home_starters, away_starters, "inferred"

## Build a reusable stint engine

This function processes one game at a time. It initializes lineups, creates stint boundaries at substitutions and period changes, updates lineup state, and logs substitution anomalies.

In [10]:
def build_stints_for_game(game_df):
    game_df = game_df.sort_values(["period", "eventnum"]).copy()

    game_id = game_df["game_id"].iloc[0]

    home_team_id, away_team_id = infer_home_away_team_ids(game_df)

    if home_team_id is None or away_team_id is None:
        return pd.DataFrame(), pd.DataFrame(), {
            "game_id": game_id,
            "status": "failed_team_inference"
        }

    home_starters, away_starters, starter_status = infer_starters(
        game_df,
        home_team_id,
        away_team_id
    )

    if home_starters is None or away_starters is None:
        return pd.DataFrame(), pd.DataFrame(), {
            "game_id": game_id,
            "status": starter_status
        }

    current_home_lineup = set(map(str, home_starters))
    current_away_lineup = set(map(str, away_starters))

    stints = []
    sub_anomalies = []

    first_row = game_df.iloc[0]

    current_stint = {
        "game_id": game_id,
        "home_team_id": home_team_id,
        "away_team_id": away_team_id,
        "period": int(first_row["period"]),
        "start_eventnum": int(first_row["eventnum"]),
        "start_clock": first_row["pctimestring"],
        "home_lineup": sorted(current_home_lineup),
        "away_lineup": sorted(current_away_lineup),
        "start_home_score": int(first_row["home_score"]),
        "start_away_score": int(first_row["away_score"])
    }

    for idx in range(len(game_df) - 1):
        row = game_df.iloc[idx]
        next_row = game_df.iloc[idx + 1]

        event_type = int(row["eventmsgtype"])
        eventnum = int(row["eventnum"])
        clock = row["pctimestring"]

        boundary = event_type in [8, 12, 13]

        if boundary:
            current_stint["end_eventnum"] = eventnum
            current_stint["end_clock"] = clock
            current_stint["end_home_score"] = int(row["home_score"])
            current_stint["end_away_score"] = int(row["away_score"])

            current_stint["home_points"] = (
                current_stint["end_home_score"] - current_stint["start_home_score"]
            )

            current_stint["away_points"] = (
                current_stint["end_away_score"] - current_stint["start_away_score"]
            )

            if current_stint["start_eventnum"] != current_stint["end_eventnum"]:
                stints.append(current_stint.copy())

            if event_type == 8:
                team_id = str(row["player1_team_id"])
                player_out = str(row["player1_id"])
                player_in = str(row["player2_id"])

                if team_id == home_team_id:
                    lineup = current_home_lineup
                    team_label = "home"
                elif team_id == away_team_id:
                    lineup = current_away_lineup
                    team_label = "away"
                else:
                    lineup = None
                    team_label = "unknown"

                if lineup is not None:
                    before_lineup = sorted(lineup)

                    removed = player_out in lineup
                    added = player_in not in lineup

                    if removed:
                        lineup.remove(player_out)

                    if added:
                        lineup.add(player_in)

                    if len(lineup) < 5 and removed and not added:
                        lineup.add(player_out)

                    if len(lineup) > 5:
                        if player_out in lineup and player_in in lineup:
                            lineup.discard(player_out)

                        while len(lineup) > 5:
                            lineup.remove(sorted(lineup)[-1])

                    after_lineup = sorted(lineup)

                    if removed and added:
                        action_taken = "applied_clean"
                    elif removed and not added:
                        action_taken = "removed_only"
                    elif not removed and added:
                        action_taken = "added_only"
                    else:
                        action_taken = "no_change"

                    sub_anomalies.append({
                        "game_id": game_id,
                        "eventnum": int(row["eventnum"]),
                        "period": int(row["period"]),
                        "pctimestring": row["pctimestring"],
                        "team_label": team_label,
                        "team_id": team_id,
                        "player_out": player_out,
                        "player_in": player_in,
                        "removed": removed,
                        "added": added,
                        "lineup_size_after": len(lineup),
                        "action_taken": action_taken,
                        "before_lineup": before_lineup,
                        "after_lineup": after_lineup
                    })

            current_stint = {
                "game_id": game_id,
                "home_team_id": home_team_id,
                "away_team_id": away_team_id,
                "period": int(next_row["period"]),
                "start_eventnum": int(next_row["eventnum"]),
                "start_clock": next_row["pctimestring"],
                "home_lineup": sorted(current_home_lineup),
                "away_lineup": sorted(current_away_lineup),
                "start_home_score": int(next_row["home_score"]),
                "start_away_score": int(next_row["away_score"])
            }

    last_row = game_df.iloc[-1]

    current_stint["end_eventnum"] = int(last_row["eventnum"])
    current_stint["end_clock"] = last_row["pctimestring"]
    current_stint["end_home_score"] = int(last_row["home_score"])
    current_stint["end_away_score"] = int(last_row["away_score"])

    current_stint["home_points"] = (
        current_stint["end_home_score"] - current_stint["start_home_score"]
    )

    current_stint["away_points"] = (
        current_stint["end_away_score"] - current_stint["start_away_score"]
    )

    if current_stint["start_eventnum"] != current_stint["end_eventnum"]:
        stints.append(current_stint.copy())

    return (
        pd.DataFrame(stints),
        pd.DataFrame(sub_anomalies),
        {
            "game_id": game_id,
            "status": "processed",
            "home_team_id": home_team_id,
            "away_team_id": away_team_id,
            "home_starters": home_starters,
            "away_starters": away_starters
        }
    )

## Process the multi-game sample

Each game is processed independently. Valid stint outputs are combined into one multi-game stint dataset, while game-level status information is logged for transparency.

In [11]:
all_stints = []
all_anomalies = []
game_logs = []

for game_id, game_df in pbp.groupby("game_id"):
    stints_game, anomalies_game, log = build_stints_for_game(game_df)

    game_logs.append(log)

    if not stints_game.empty:
        all_stints.append(stints_game)

    if not anomalies_game.empty:
        all_anomalies.append(anomalies_game)

stints_df = pd.concat(all_stints, ignore_index=True) if all_stints else pd.DataFrame()
sub_anomalies_df = pd.concat(all_anomalies, ignore_index=True) if all_anomalies else pd.DataFrame()
game_logs_df = pd.DataFrame(game_logs)

stints_df.shape, sub_anomalies_df.shape, game_logs_df["status"].value_counts()

C:\Users\jmontanez\AppData\Local\Temp\ipykernel_20264\1262074007.py:62: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)
  .apply(lambda x: set(x.dropna().astype(str)))
C:\Users\jmontanez\AppData\Local\Temp\ipykernel_20264\1262074007.py:62: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)
  .apply(lambda 

((6293, 16),
 (7625, 14),
 processed    200
 Name: status, dtype: int64)

## Substitution anomaly interpretation

Most substitutions were applied cleanly, while some required tolerant handling because the event stream did not perfectly align with the inferred lineup state.

These cases are expected in real play-by-play data and are logged for future review. Importantly, they do not compromise lineup integrity in the final stint dataset.

## Validation and quality checks

##  Validate multi-game stint output

We verify that the reconstructed stints preserve five-player lineup integrity and add player names to improve readability.

In [12]:
stints_df["home_n_players"] = stints_df["home_lineup"].apply(len)
stints_df["away_n_players"] = stints_df["away_lineup"].apply(len)

stints_df["home_lineup_names"] = stints_df["home_lineup"].apply(lineup_ids_to_names)
stints_df["away_lineup_names"] = stints_df["away_lineup"].apply(lineup_ids_to_names)

stints_df[["home_n_players", "away_n_players"]].value_counts()

home_n_players  away_n_players
5               5                 6293
dtype: int64

In [13]:
game_logs_df["status"].value_counts()

processed    200
Name: status, dtype: int64

In [14]:
sub_anomalies_df["action_taken"].value_counts()

applied_clean    4014
removed_only     1370
added_only       1323
no_change         918
Name: action_taken, dtype: int64

## Multi-game validation interpretation

The multi-game stint engine successfully processed all 50 selected regular-season games.

All reconstructed stints preserve five-player lineup integrity for both teams, confirming that the generalized substitution logic scales beyond the original one-game prototype.

Substitution events that required tolerant handling are logged separately, allowing the pipeline to remain stable while preserving transparency around noisy event cases.

## Save multi-game stint datasets

The multi-game stint dataset, substitution anomaly log, and game-level processing log are saved for downstream validation, modeling, and dashboard development.

In [15]:
stints_df.to_pickle("../data/interim/stints_multigame.pkl")
stints_df.to_csv("../data/interim/stints_multigame.csv", index=False)

sub_anomalies_df.to_pickle("../data/interim/sub_anomalies_multigame.pkl")
sub_anomalies_df.to_csv("../data/interim/sub_anomalies_multigame.csv", index=False)

game_logs_df.to_csv("../data/interim/game_processing_logs.csv", index=False)

print("Multi-game stint outputs saved successfully.")

Multi-game stint outputs saved successfully.


## Key findings

This notebook generalizes the original one-game stint reconstruction prototype into a scalable multi-game pipeline.

The pipeline now:

- loads a regular-season multi-game sample
- processes each game independently
- infers teams and starting lineups automatically
- tracks lineup changes through substitutions
- logs substitution anomalies and game-level processing status
- produces a combined multi-game stint dataset

This expanded dataset provides a stronger foundation for lineup metrics, player impact modeling, and predictive analysis than the original single-game prototype.

## Next step

The next stage is to update the validation and metric notebooks to consume the new multi-game stint dataset instead of the original one-game prototype..